# Understanding LSTMs: A Practical Guide

This notebook provides a clear explanation of Long Short-Term Memory (LSTM) networks, their components, and how they can be used for a simple time series prediction task.

-----

## 🧐 What is an LSTM?

An **LSTM** is a special kind of **Recurrent Neural Network (RNN)** designed to learn from sequences of data and remember information over long periods. Standard RNNs struggle with the **vanishing gradient problem**, where the network's ability to learn from data far back in the sequence diminishes. LSTMs solve this by using a sophisticated internal structure called a **cell state**.

The key idea behind an LSTM is its ability to selectively remember or forget information. It's like a conveyor belt carrying information through time. The LSTM's "gates" control what information is added to or removed from this conveyor belt.

-----

## ⚙️ The Inner Workings: LSTM Gates

The power of an LSTM comes from its three main gates, which are essentially neural networks themselves.

  * **Forget Gate ($f_t$)**: This gate decides what information to throw away from the cell state. It looks at the current input ($x_t$) and the previous hidden state ($h_{t-1}$) and outputs a number between 0 and 1 for each number in the cell state. A value of 0 means "completely forget this," while a value of 1 means "completely keep this."

  * **Input Gate ($i_t$)**: This gate decides which new information to store in the cell state. It has two parts:

    1.  A sigmoid layer that decides which values to update.
    2.  A `tanh` layer that creates a vector of new candidate values ($\tilde{C}_t$) to add to the state.

  * **Output Gate ($o_t$)**: This gate decides what part of the cell state to output as the new hidden state ($h_t$). It uses a sigmoid layer to choose which parts of the cell state to output, and then the cell state itself is put through a `tanh` function. The result is then multiplied by the output of the sigmoid layer to get the final hidden state.

These gates work together to update the **cell state ($C_t$)** at each time step, allowing the network to retain crucial long-term dependencies.

-----


## 📈 Example: Simple Time Series Prediction

Let's use a simple sine wave to demonstrate how an LSTM can predict future values in a sequence. We'll generate a sequence of numbers and train an LSTM to predict the next number in the sequence.


### **Step 1: Setup and Data Generation**

First, let's import the necessary libraries and create our sine wave data.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import matplotlib.pyplot as plt

# Generate a simple sine wave sequence
np.random.seed(42)
sequence_length = 500
t = np.linspace(0, 50, sequence_length)
data = np.sin(t) + np.random.randn(sequence_length) * 0.1 # Add some noise

plt.figure(figsize=(10, 6))
plt.plot(data)
plt.title("Noisy Sine Wave Data")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.show()

### **Step 2: Prepare the Data for LSTM**

LSTMs require input data in a specific 3D format: `(samples, timesteps, features)`.

  * `samples`: The number of data sequences.
  * `timesteps`: The length of each input sequence.
  * `features`: The number of features at each time step (for our example, this is 1).

We will create sequences where the LSTM is given a window of past values to predict the next value.


In [ ]:
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        # The input sequence is the data from i to i + n_steps
        X.append(data[i:i+n_steps])
        # The target is the value right after the input sequence
        y.append(data[i+n_steps])
    return np.array(X), np.array(y)

n_steps = 10 # Number of past time steps to consider
X, y = create_sequences(data, n_steps)

# Reshape X to be 3D: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

print("Shape of X (Input sequences):", X.shape)
print("Shape of y (Target values):", y.shape)

### **Step 3: Build the LSTM Model**

We'll use a `Sequential` model from Keras. The `LSTM` layer is the core of our model. We'll add a `Dense` layer at the end to output a single prediction.


In [ ]:
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(n_steps, 1)))
model.add(Dense(1)) # Output layer for a single prediction

model.compile(optimizer='adam', loss='mse')

model.summary()

### **Step 4: Train the Model**

Now, let's train our LSTM on the prepared data.


In [ ]:
history = model.fit(X, y, epochs=100, verbose=1)

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'])
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

### **Step 5: Make Predictions**

Finally, let's see how our trained model performs on some of the data.

In [ ]:
# Use the last part of the original data to make predictions
test_data = data[-50:]
test_input, _ = create_sequences(test_data, n_steps)
test_input = test_input.reshape(test_input.shape[0], test_input.shape[1], 1)

predictions = model.predict(test_input)

# Plot the actual vs. predicted values
plt.figure(figsize=(10, 6))
plt.plot(test_data[n_steps:], label='Actual Values')
plt.plot(predictions, label='Predicted Values')
plt.title('Actual vs. Predicted Time Series')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.show()

The plot shows how the LSTM model has learned the underlying pattern of the sine wave and is able to predict the future values with reasonable accuracy.